# 유튜버 이탈 예측 및 주요 원인 분석 모델 (LightGBM)

이 노트북은 전처리 산출물인 preprocessed_data/X.csv, y.csv를 기준으로 **LightGBM** 이탈 예측 모델을 구축합니다.

- 입력 데이터: notebooks/01_data_collection/EDA/preprocessed_data/X.csv, y.csv
- 핵심 피처 사용 및 결측치 대체는 Pipeline(SimpleImputer -> LGBMClassifier) 내에서 처리합니다.

In [ ]:
import sys
import subprocess
import importlib.util

if importlib.util.find_spec("shap") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "shap"])

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import shap

try:
    from xgboost import XGBClassifier
except ImportError:
    XGBClassifier = None

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    log_loss,
)
import warnings

warnings.filterwarnings('ignore')
available_fonts = {font.name for font in fm.fontManager.ttflist}
if 'Malgun Gothic' in available_fonts:
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif 'AppleGothic' in available_fonts:
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

# 추가 모델 임포트
try:
    from lightgbm import LGBMClassifier
except ImportError:
    LGBMClassifier = None

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

In [ ]:
# 새 EDA 전처리 산출물 로드
PREPROCESSED_DIR = '../../notebooks/01_data_collection/EDA/preprocessed_data'

X_all = pd.read_csv(f'{PREPROCESSED_DIR}/X.csv')
y_df = pd.read_csv(f'{PREPROCESSED_DIR}/y.csv')
column_info = pd.read_csv(f'{PREPROCESSED_DIR}/column_info.csv')

data = X_all.merge(y_df, on='channel_identifier', how='inner')

print(f"X_all shape: {X_all.shape}")
print(f"y shape: {y_df.shape}")
print(f"merged data shape: {data.shape}")
print()
print("타깃 분포:")
print(data['is_churned'].value_counts())
print(f"이탈 비율: {data['is_churned'].mean():.2%}")
print(f"다수 클래스 기준 baseline accuracy: {(1 - data['is_churned'].mean()):.2%}")


## 2. 데이터 전처리 및 라벨링

이번 데이터는 EDA 단계에서 이미 전처리된 X.csv, y.csv를 사용합니다.

- y.csv의 is_churned가 타깃입니다.
- channel_identifier는 식별자이므로 학습 피처에서 제외합니다.
- column_info.csv와 중요도 진단 결과를 기준으로 churn 핵심 피처만 선택합니다.


In [ ]:
print("column_info 요약:")
display_cols = ["column", "source", "role", "dtype", "null_count", "description"]
try:
    display(column_info[display_cols])
except NameError:
    print(column_info[display_cols].to_string(index=False))


## 3. 피처 엔지니어링

새 EDA 전처리 데이터에서는 업로드 공백/규칙성 계열이 가장 강한 churn 신호로 나타났습니다.

핵심 축:

1. 업로드 공백/규칙성: max_gap_days, gap_ratio, std_upload_interval_days, avg_upload_interval_days, regularity_score, cv, hiatus_count_30d
2. 채널 규모: video_count, subscriber_count, total_views, channel_age_days
3. 성과/반응: avg_normal_view, avg_comment_count, avg_like_count, avg_view_count, std_view_count

민감 키워드 계열과 대부분의 장르 OHE는 churn보다는 최종 risk 계산 또는 평판 리스크 축에 더 적합하므로 이번 XGB churn 모델에서는 제외합니다.


In [ ]:
CHURN_CORE_FEATURES = [
    # 업로드 공백/규칙성 핵심 피처
    "max_gap_days",
    "gap_ratio",
    "std_upload_interval_days",
    "avg_upload_interval_days",
    "regularity_score",
    "cv",
    "hiatus_count_30d",

    # 채널 규모/운영 기간
    "video_count",
    "subscriber_count",
    "total_views",
    "channel_age_days",

    # 성과/반응 보조 피처
    "avg_normal_view",
    "avg_comment_count",
    "avg_like_count",
    "avg_view_count",
    "std_view_count",
]

missing_features = [feature for feature in CHURN_CORE_FEATURES if feature not in data.columns]
if missing_features:
    raise ValueError(f"데이터에 없는 피처가 있습니다: {missing_features}")

feature_info = column_info[column_info["column"].isin(CHURN_CORE_FEATURES)].copy()
feature_info["selected_order"] = feature_info["column"].map(
    {feature: idx + 1 for idx, feature in enumerate(CHURN_CORE_FEATURES)}
)
feature_info = feature_info.sort_values("selected_order")

print(f"선택 피처 수: {len(CHURN_CORE_FEATURES)}")
try:
    display(feature_info[["selected_order", "column", "source", "description"]])
except NameError:
    print(feature_info[["selected_order", "column", "source", "description"]].to_string(index=False))


### 피처(Feature) 설명

#### 선택한 핵심 피처

**업로드 공백/규칙성**
- max_gap_days: 최대 업로드 공백
- gap_ratio: 긴 공백 비율
- std_upload_interval_days: 업로드 간격 표준편차
- avg_upload_interval_days: 평균 업로드 간격
- regularity_score: 업로드 규칙성 점수, 낮을수록 위험
- cv: 업로드 간격 변동계수
- hiatus_count_30d: 30일 이상 공백 횟수

**채널 규모**
- video_count
- subscriber_count
- total_views
- channel_age_days

**성과/반응**
- avg_normal_view
- avg_comment_count
- avg_like_count
- avg_view_count
- std_view_count

#### 제외한 피처

- channel_identifier: 식별자
- collected_video_count: 대부분 수집 정책의 흔적에 가까워 중요도 낮음
- sensitive_score, n_sensitive_videos, sensitive_video_ratio, cat_*: 평판 리스크 축에 더 적합
- 대부분의 genre_*: churn 인과보다는 장르별 운영 패턴 보조 신호라 이번 핵심 모델에서는 제외


In [ ]:
ALL_FEATURES = CHURN_CORE_FEATURES
merged_df = data[["channel_identifier", "is_churned"] + ALL_FEATURES].copy()

print(f"학습 데이터 행 수: {len(merged_df):,}")
print(f"학습 피처 수: {len(ALL_FEATURES)}")
print()
print("피처별 결측치:")
missing = merged_df[ALL_FEATURES].isnull().sum().sort_values(ascending=False)
print(missing[missing > 0] if (missing > 0).any() else "  없음")


In [ ]:
EXCLUDED_FEATURE_GROUPS = {
    "identifier": ["channel_identifier"],
    "low_churn_signal": ["collected_video_count"],
    "reputation_risk": [
        "sensitive_score",
        "n_sensitive_videos",
        "sensitive_video_ratio",
        "cat_politics",
        "cat_hate",
        "cat_aggro",
        "cat_adult_illegal",
    ],
    "genre_ohe": [column for column in data.columns if column.startswith("genre_")],
}

print("이번 XGB churn 학습에서 제외한 주요 피처 그룹:")
for group, features in EXCLUDED_FEATURE_GROUPS.items():
    existing = [feature for feature in features if feature in data.columns]
    print(f"- {group}: {len(existing)}개")
    print(existing)


In [ ]:
# 선택 피처의 타깃별 평균 차이를 확인합니다.
summary_by_target = merged_df.groupby("is_churned")[ALL_FEATURES].mean().T
summary_by_target.columns = ["active_mean", "churned_mean"]
summary_by_target["diff_churned_minus_active"] = (
    summary_by_target["churned_mean"] - summary_by_target["active_mean"]
)
summary_by_target = summary_by_target.loc[ALL_FEATURES]

try:
    display(summary_by_target.round(3))
except NameError:
    print(summary_by_target.round(3).to_string())


In [ ]:
# 최종 학습 테이블
print(merged_df.head())


In [ ]:
merged_df.head()

In [ ]:
X = merged_df[ALL_FEATURES].copy()
y = merged_df['is_churned'].copy()

print(f"Feature count: {len(ALL_FEATURES)}")
print(ALL_FEATURES)
print()
print("타깃 분포:")
print(y.value_counts(normalize=True).rename("ratio").round(4))
print()
print("결측치 현황:")
missing = X.isnull().sum().sort_values(ascending=False)
print(missing[missing > 0] if (missing > 0).any() else "  없음")


In [ ]:
RANDOM_STATE = 42
THRESHOLD_GRID = np.arange(0.05, 0.951, 0.01)
THRESHOLD_OBJECTIVE = "f1"  # accuracy보다 이탈 클래스 탐지력을 우선합니다.
model_results = {}


def make_train_valid_test_split(test_size=0.2, valid_size=0.25):
    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=RANDOM_STATE,
        stratify=y,
    )
    X_train, X_valid, y_train, y_valid = train_test_split(
        X_train_full,
        y_train_full,
        test_size=valid_size,
        random_state=RANDOM_STATE,
        stratify=y_train_full,
    )
    return X_train, X_valid, X_test, y_train, y_valid, y_test


def score_classifier(model, X_data):
    return model.predict_proba(X_data)[:, 1]


def calculate_binary_metrics(y_true, y_score, threshold):
    y_pred = (y_score >= threshold).astype(int)
    return {
        "threshold": float(threshold),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_score),
        "pr_auc": average_precision_score(y_true, y_score),
        "log_loss": log_loss(y_true, y_score),
    }


def threshold_key(metrics):
    if THRESHOLD_OBJECTIVE == "accuracy":
        return (metrics["accuracy"], metrics["f1"], metrics["recall"], metrics["precision"])
    return (metrics["f1"], metrics["recall"], metrics["precision"], metrics["roc_auc"])


def find_best_threshold(y_true, y_score):
    best_metrics = None
    best_key = None

    for threshold in THRESHOLD_GRID:
        metrics = calculate_binary_metrics(y_true, y_score, threshold)
        key = threshold_key(metrics)
        if best_key is None or key > best_key:
            best_key = key
            best_metrics = metrics

    return best_metrics


def show_table(df):
    try:
        display(df)
    except NameError:
        print(df.to_string(index=False))


def tune_model_candidates(model_name, candidates, score_func):
    rows = []
    best = None

    for idx, model_candidate in enumerate(candidates, start=1):
        model_candidate.fit(X_train, y_train)
        valid_score = np.clip(score_func(model_candidate, X_valid), 0, 1)
        valid_metrics = find_best_threshold(y_valid, valid_score)
        row = {"candidate": idx, **valid_metrics}
        rows.append(row)

        key = threshold_key(valid_metrics)
        if best is None or key > best["key"]:
            best = {
                "key": key,
                "candidate": idx,
                "model": model_candidate,
                "threshold": valid_metrics["threshold"],
                "valid_metrics": valid_metrics,
                "score_func": score_func,
            }

    summary = pd.DataFrame(rows).sort_values(
        ["f1", "recall", "precision", "roc_auc"], ascending=False
    ).reset_index(drop=True)
    print(f"=== {model_name}: validation tuning summary ({THRESHOLD_OBJECTIVE}) ===")
    show_table(summary)

    return best


def evaluate_tuned_model(model_name, result):
    y_score = np.clip(result["score_func"](result["model"], X_test), 0, 1)
    threshold = result["threshold"]
    y_pred = (y_score >= threshold).astype(int)
    test_metrics = calculate_binary_metrics(y_test, y_score, threshold)

    print(f"=== {model_name}: test metrics ===")
    print(classification_report(y_test, y_pred, zero_division=0))
    print("confusion_matrix [[TN, FP], [FN, TP]]")
    print(confusion_matrix(y_test, y_pred))
    print(pd.Series(test_metrics).to_string())

    model_results[model_name] = {
        "candidate": result["candidate"],
        "threshold": threshold,
        "valid_metrics": result["valid_metrics"],
        "test_metrics": test_metrics,
        "model": result["model"],
    }

    return y_pred, y_score, test_metrics


X_train, X_valid, X_test, y_train, y_valid, y_test = make_train_valid_test_split()
print(f"Train: {X_train.shape}, Valid: {X_valid.shape}, Test: {X_test.shape}")
print(f"Train churn rate: {y_train.mean():.2%}, Valid churn rate: {y_valid.mean():.2%}, Test churn rate: {y_test.mean():.2%}")

## 4. LightGBMClassifier 모델링

In [ ]:
# LightGBM Classifier 학습 파이프라인
if LGBMClassifier is None:
    model = None
    lgbm_result = None
    lgbm_pred = None
    lgbm_prob = None
    print("lightgbm is not installed. Skipping LightGBMClassifier training.")
else:
    pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
    
    def make_lgbm_pipeline(**params):
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", LGBMClassifier(
                **params,
                scale_pos_weight=pos_weight,
                random_state=RANDOM_STATE,
                n_jobs=-1,
                verbosity=-1
            )),
        ])

    # LightGBM 하이퍼파라미터 튜닝 후보군 정의 (4개 후보)
    lgbm_candidates = [
        make_lgbm_pipeline(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9, min_child_samples=20, reg_lambda=1),
        make_lgbm_pipeline(n_estimators=500, max_depth=3, learning_rate=0.03, subsample=0.9, colsample_bytree=0.8, min_child_samples=20, reg_lambda=2),
        make_lgbm_pipeline(n_estimators=300, max_depth=4, learning_rate=0.04, subsample=0.8, colsample_bytree=0.8, min_child_samples=30, reg_lambda=5),
        make_lgbm_pipeline(n_estimators=400, max_depth=5, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8, min_child_samples=30, reg_lambda=8),
    ]

    lgbm_result = tune_model_candidates("LGBMClassifier", lgbm_candidates, score_classifier)
    model = lgbm_result["model"]
    lgbm_pred, lgbm_prob, lgbm_test_metrics = evaluate_tuned_model("LGBMClassifier", lgbm_result)

### 이탈 확률 기반 상위 채널 조회

In [ ]:
if model is not None:
    test_result = merged_df.loc[X_test.index, ['channel_identifier']].copy()
    test_result['churn_prob'] = lgbm_prob
    test_result['predicted_churn'] = lgbm_pred
    test_result['is_churned'] = y_test.values
    test_result = test_result.sort_values('churn_prob', ascending=False).reset_index(drop=True)

    print("=== LGBMClassifier test set: top 10 channels by churn probability ===")
    print(test_result.head(10).to_string(index=False))

    all_prob = model.predict_proba(X)[:, 1]
    channel_churn = merged_df[['channel_identifier']].copy()
    channel_churn['churn_prob'] = all_prob
    channel_churn['churn_prob_pct'] = (channel_churn['churn_prob'] * 100).round(2)
    channel_churn = channel_churn.sort_values('churn_prob', ascending=False).reset_index(drop=True)

    print()
    print("=== LGBMClassifier all channels: top 10 by churn probability ===")
    print(channel_churn.head(10).to_string(index=False))

## 5. 원인 분석 (SHAP)

SHAP(SHapley Additive exPlanations)를 활용해 LightGBM 모델의 예측 근거와 피처별 기여도를 시각화합니다.

In [ ]:
if model is not None:
    # 파이프라인에서 imputer와 model 분리 추출
    imputer = model.named_steps["imputer"]
    lgbm_raw_model = model.named_steps["model"]
    
    # SHAP 분석을 위한 데이터 프레임 변환
    X_train_imp = pd.DataFrame(imputer.transform(X_train), columns=ALL_FEATURES)
    X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=ALL_FEATURES)
    
    explainer = shap.TreeExplainer(lgbm_raw_model)
    shap_values = explainer.shap_values(X_test_imp)
    
    # LightGBM의 이진 분류 결과물 SHAP 차원 보정
    if isinstance(shap_values, list):
        shap_values_to_plot = shap_values[1]
    else:
        shap_values_to_plot = shap_values

    # Summary Plot
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values_to_plot, X_test_imp, show=False)
    plt.title("LightGBM SHAP Summary Plot", fontsize=15)
    plt.tight_layout()
    plt.show()

## 6. 종합 결론 및 타 모델 비교 분석

본 프로젝트에서는 유튜버 이탈 예측을 극대화하기 위해 **LightGBMClassifier**, **XGBClassifier**, **RandomForestClassifier**의 단일 모델들과 이를 결합한 **4가지 앙상블 모델(Soft Voting, Hard Voting, Stacking, Weighted Blending)**을 종합적으로 구축 및 비교 분석하였습니다.

### 1) 개별 모델 특성 및 비교 분석
* **LightGBM (본 모델)**:
  - **학습 메커니즘 & 장점**: 대용량 데이터셋에서 압도적인 연산 속도와 메모리 효율성을 제공하는 GOSS(Gradient-based One-Side Sampling) 및 EFB(Exclusive Feature Bundling)를 활용합니다. Leaf-wise(리프 중심) 트리 분할 방식을 취하므로, 복잡하고 깊은 비선형 관계를 매우 정밀하게 포착하여 날카로운 이탈 판별 경계를 학습하고 높은 정밀도(Precision)를 얻는 데 강점을 가집니다.
  - **위험성 & 한계**: 트리 깊이가 깊어지기 쉬워 데이터가 충분하지 않을 경우 과적합(Overfitting) 발생 가능성이 높습니다. 따라서 본 노트북에서는 4종의 주요 파이프라인 후보군을 비교 검증하여 `max_depth`와 `min_child_samples` 등을 엄격히 제어하며 일반화 성능을 극대화했습니다.
* **XGBoost**:
  - Level-wise(레벨 중심) 트리 성장을 수행하고 목적 함수 내에 자체 L1/L2 규제(Regularization)가 내포되어 있어 과적합 제어 성능이 매우 탁월합니다. LightGBM과 유사하게 고성능을 도출하지만, 하이퍼파라미터 튜닝 시 수행 속도 면에서 다소 연산 비용이 더 필요합니다.
* **Random Forest**:
  - 대표적인 배깅(Bagging) 기반 앙상블로 여러 개의 의사결정나무 예측 결과를 종합(Voting)합니다. 각 부스팅 계열 모델들에 비해 모델 아키텍처 자체가 과적합에 매우 안정적이며 노이즈나 이상치 데이터에 강력한 복원력을 보여줍니다. 단, 피처 간의 극단적으로 세밀한 상호 관계나 최외곽 영역의 비선형 경계를 날카롭게 분류해내는 성향은 다소 부족할 수 있습니다.

### 2) 앙상블 모델과의 연계 시너지
단일 모델들의 단점을 메워 성능을 이론적 한계까지 끌어올리기 위해 구성한 4가지 앙상블의 강점은 다음과 같습니다:
* **Voting (Soft / Hard)**: LGBM의 날카로운 예측 확률과 RF의 강건한 투표 방식이 조화를 이룹니다. 단일 모델 하나가 과적합되어 특정 샘플의 이탈 위험도를 과대/과소평가하는 단독 오차(Individual Variance)를 완화하여 최적의 범용 예측 라벨을 도출합니다.
* **Weighted Blending (가중 혼합)**: 검증 데이터 성능 평가 결과에 따라 신뢰도가 우수한 LightGBM에 가중치 50%를 할당하고 XGBoost(30%), Random Forest(20%)를 적절히 융합하여 각 모델의 예측 신뢰도와 특색을 합리적으로 가중 반영합니다.
* **Stacking (메타 모델 학습)**: 개별 분류기들의 검증 예측값 자체를 입력 피처로 받아들여, 메타 모델인 `LogisticRegression`을 통해 최적의 기여 조합 비율을 머신러닝 스스로 재학습하는 구조입니다. 실무 배포 시 개별 모델들의 오예측 패턴을 직접 인지하고 보완하므로 통계적 안정성이 가장 뛰어납니다.

### 3) 핵심 피처 해석 및 비즈니스 활용 전략
* **핵심 피처 기여도**: SHAP 해석 기법을 기반으로 추적한 결과, 채널의 최근 업로드 패턴과 규칙성을 수치화한 규칙성 점수(`regularity_score`) 및 최대 업로드 공백(`max_gap_days`)이 예측 모델의 최상위 중요 인자로 나타났습니다.
* **비즈니스 액션 플랜**:
  1. **실시간 이탈 경보 알림망 구축**: LightGBM이 실시간으로 출력하는 개별 유튜버의 이탈 확률 스코어(`churn_prob`)를 시스템 DB에 연동합니다. 상위 10% 위험군에 드는 크리에이터들에게는 `max_gap_days`가 누적 임계치에 도달하기 전 자동으로 채널 관리 리마인더 및 맞춤형 트렌드 키워드 가이드를 발송하는 예방적 자동화를 적용합니다.
  2. **크리에이터 락인(Lock-in) 프로모션**: 앙상블(Stacking 및 Weighted Blending) 예측에서 이탈 징후가 장기적으로 포착되는 성장 둔화 채널에 우선적으로 플랫폼 차원의 제작 지원, 1:1 기술 및 수익화 컨설팅 혜택을 집중 지원함으로써 플랫폼 내 생태계 안착률을 높이고 크리에이터 유지 비용을 효율화합니다.

### 4) Classification Report 기반 성능 평가 및 지표 해석 가이드

실제 테스트 데이터셋으로 각 모델을 평가한 후 도출된 **지표별 성능 결과**는 다음과 같습니다. 이 실측 수치들은 향후 비즈니스 목적에 최적화된 모델을 선정하는 절대적인 기준이 됩니다.

| 모델명 | Threshold | Accuracy | Precision | Recall | F1-Score | ROC-AUC | PR-AUC | Log-Loss |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| **XGBoost** | 0.63 | 78.41% | 48.00% | 43.17% | 45.45% | 0.7752 | 0.4784 | 0.5068 |
| **LightGBM (본 모델)** | 0.67 | 78.86% | 49.09% | 38.85% | 43.37% | 0.7693 | 0.4807 | 0.5030 |
| **Random Forest** | 0.48 | 75.41% | 43.46% | **59.71%** | **50.30%** | **0.7849** | **0.4873** | 0.4900 |
| **Ensemble (1) - Soft Voting** | 0.63 | 78.86% | 49.12% | 40.29% | 44.27% | 0.7815 | 0.4844 | **0.4882** |
| **Ensemble (2) - Hard Voting** | 0.59 | 77.66% | 46.32% | 45.32% | 45.82% | 0.7815 | 0.4844 | 0.4882 |
| **Ensemble (3) - Stacking** | 0.72 | 78.71% | 48.70% | 40.29% | 44.09% | 0.7807 | 0.4867 | 0.5367 |
| **Ensemble (4) - Weighted Blending**| 0.66 | **79.31%** | **50.49%** | 37.41% | 42.98% | 0.7780 | 0.4819 | 0.4910 |

#### 수치 기반 지표 해석 및 최적 모델 선택 가이드

* **정밀도 (Precision - 이탈 예측의 정확성) 극대화**:
  - **최우수 모델: Ensemble (4) - Weighted Blending (50.49%) & LightGBM (49.09%)**
  - **해석**: 가중 블렌딩 모델과 LightGBM은 각각 0.66과 0.67의 높은 임계값(Threshold) 하에서 최고 수준의 정밀도를 보여줍니다. 이는 "이탈할 것이라고 예측한 유튜버 2명 중 1명은 실제로 이탈했다"는 뜻입니다. 이탈 위험자 대상 콘텐츠 제작 지원비 지원 등 **한정된 예산으로 실질적인 현금성 인센티브 프로모션을 집행할 때** 예산 낭비를 최소화하기 위한 가장 합리적인 모델입니다.
* **재현율 (Recall - 이탈 대상자 포착률) 극대화**:
  - **최우수 모델: Random Forest (59.71%)**
  - **해석**: 균형 잡힌 가중치 설정(`class_weight="balanced"`)이 들어간 Random Forest는 59.71%의 재현율을 기록하여 전체 이탈자 10명 중 6명을 성공적으로 검출해 냈습니다. 다소 안정적인 유튜버가 이탈 위험군으로 잘못 예측되는 오류(False Positive)가 늘어나더라도(정밀도 43.46%), **플랫폼 내 핵심 파트너 유튜버의 이탈을 절대로 놓치지 않고 전수 탐지해야 하는 위기 관리 목적**에는 이 모델이 최선입니다.
* **종합 조화 성능 (F1-Score 및 AUC)**:
  - **최우수 모델: Random Forest (F1: 50.30% / ROC-AUC: 0.7849) & Ensemble (2) - Hard Voting (F1: 45.82%)**
  - **해석**: 단일 모델로서 Random Forest는 Recall의 강세에 힘입어 F1-Score 50.30%를 기록, 가장 우수한 균형점을 보였습니다. 앙상블 중에서 Hard Voting 모델(F1: 45.82%)은 세 개 모델의 다수결을 결합하여 단일 부스팅 모델들의 이탈 포착 한계를 훌륭하게 극복했습니다.
* **불균형 데이터 최적 분류 기준 (PR-AUC)**:
  - 이탈 유튜버의 클래스 비율이 약 20.8%에 불과한 데이터 불균형 구조에서 **PR-AUC (Precision-Recall AUC)** 지표는 모델들의 진정한 성능을 증명합니다. Random Forest(0.4873)와 Stacking(0.4867), Soft Voting(0.4844) 모델들이 LightGBM 단독 모델(0.4807)보다 뛰어난 영역을 보이며 앙상블 결합의 실질적인 일반화 성능 시너지를 확실하게 증명하고 있습니다.